# Семинар I. Нейросети и ДНК: классификация функциональных элементов генома

*Автор:* Боревский Андрей

---
## Оглавление
1. [Введение](#введение)
2. [Подготовка данных](#подготовка-данных)
3. [Кодирование последовательностей](#кодирование-последовательностей)
4. [Архитектуры моделей](#построение-моделей)
5. [Обучение и оптимизация](#обучение-и-оптимизация)

## Введение  <a name="введение"></a>  

В этом ноутбуке мы рассмотрим применение глубокого обучения для анализа последовательностей ДНК. Наша цель - создать модель, способную классифицировать функциональные элементы генома (промоторы и энхансеры) на основе их последовательностей.

### Теоретическая справка: основы молекулярной генетики  

**1. Структура ДНК**  
ДНК состоит из:  
- **Нуклеотидов** (A, T, C, G)  
- **Сахарно-фосфатного остова**  
- Комплементарных цепей (A-T, C-G)  

**2. Функциональные элементы генома**  

| Элемент       | Функция                          | Длина (п.н.) |  
|---------------|----------------------------------|-------------|  
| **Промотор**| Инициация транскрипции           | 100-1000    |  
| **Энхансер**| Усиление экспрессии генов        | 50-1500     |  
| **Инсулятор** | Блокировка взаимодействия элементов | 200-3000  |  



**3. Биологические процессы**   

- **Транскрипция**: ДНК → РНК  
- **Трансляция**: РНК → белок  
- **Промоторы**: участки ДНК, необходимые для инициации транскрипции.  
- **Энхансеры**: регуляторные элементы, усиливающие экспрессию генов.

<img src="https://images.slideplayer.com/35/10480794/slides/slide_35.jpg" alt="Функциональные элементы" style="width:600px;height:400px;">

Нейросетевые подходы особенно полезны для:
1. Автоматического извлечения признаков из последовательностей
2. Учета сложных, нелинейных зависимостей
3. Масштабирования на большие объемы геномных данных

### Описание проекта
В этом ноутбуке мы разберём полный pipeline анализа ДНК с помощью нейросетей:

>1. Обработка BED/FASTA-файлов → извлечение последовательностей.
>
>2. Кодирование ДНК (one-hot, k-mer frequency, embeddings).
>
>3. Построение датасета для PyTorch.
>
>4. Обучение моделей (CNN, LSTM, Transformer) и их сравнение.
>
>5. Оптимизация (LR scheduling, Grad-CAM для интерпретации).

---

### Задачи на взаимопонимание

**Задача 1. Основы ДНК**  
1. Какие пары нуклеотидов образуют водородные связи в ДНК?  
2. Почему промоторы обычно расположены перед геном?  
3. Как энхансеры могут влиять на гены на расстоянии?  

**Задача 2. Биоинформатика**  
1. Какая информация хранится в BED-файле?

BED-файл хранит координаты и аннотации геномных регионов в табличном текстовом виде: хромосома, начало и конец интервала, плюс (опционально) имя, score, цепь и др.

2. Чем отличается FASTA от FASTQ форматов?  

Оба — текстовые форматы последовательностей, но FASTQ добавляет качества.

- FASTA:  
  - Строка заголовка начинается с `>` и содержит идентификатор/описание.  
  - Далее одна или несколько строк с последовательностью (ДНК/РНК/белок), без информации о качестве.  

- FASTQ:  
  - Строка 1: `@` + идентификатор.  
  - Строка 2: последовательность.  
  - Строка 3: `+` (иногда с повтором ID).  
  - Строка 4: строка ASCII‑символов с кодированием качества (обычно Phred) для каждого нуклеотида.  

Кратко: **FASTA** – только последовательность, **FASTQ** – последовательность + позначные оценки качества чтения.

3. Почему one-hot encoding подходит для ДНК?  

---


## 1. Подготовка данных  <a name="подготовка-данных"></a>

### Теоретическая справка: форматы данных  

**BED (Browser Extensible Data)**  
```chr1    100    200    Promoter_1    0    +```  
- **Поля**: хромосома, start, end, имя, score, strand  

**FASTA**  
```
>chr1:100-200  
ATCG...
```  

**Пример обработки:**  
```python
import pybedtools

# Создание BedTool объекта
promoters = pybedtools.BedTool("promoters.bed")

# Извлечение последовательностей
seqs = promoters.sequence(fi="hg38.fa")
```

**Дополнительные ресурсы:**  
- [ENCODE Project](https://www.encodeproject.org/)  
- [PyTorch для биоинформатики](https://pytorch.org/tutorials/beginner/deep_learning_dna.html)
- 
### Загрузка и предварительная обработка

> * Задача:
>   * Загрузить BED-файлы с промоторами (class=1) и энхансерами (class=0), извлечь последовательности в формате FASTA.
>
>
> * Методы:
>   * `pybedtools` — для работы с геномными координатами.
>   * `pandas` — для удобного хранения меток.
>
>
> * Используем данные из ENCODE проекта:
>   * BED-файлы с координатами элементов
>   * Референсный геном (hg38) в формате FASTA


### Программные команды

#### `datasets` (Hugging Face)
- Загрузка предобработанных геномных датасетов
- Особенности:
  - Автоматическое кэширование (`cache_dir`)
  - Поддерка удаленного кода (`trust_remote_code=True`)
  - Фильтрация по длине последовательности (`sequence_length`)

#### `Bio.Seq`
- Обработка биологических последовательностей
- Методы:
  - `Seq(sequence).upper()` - нормализация регистра
  - `reverse_complement()` - обратная цепь ДНК

#### `torch.utils.data`
- Классы для работы с данными:
  - **Dataset**: Абстракция для хранения пар (features, labels)
  - **DataLoader**: Пакетная загрузка с shuffling/sampling
  - **random_split**: Разделение на train/val

### Этапы обработки данных

1. **Загрузка аннотаций**:
   ```python
   load_dataset(task_name="regulatory_element_[promoter|enhancer]")
   ```

2. **Создание меток**:
   - Промотор → 1
   - Не промотор → 0

3. **Балансировка классов**:
   - Одинаковое количество примеров для каждого класса
   - Стратифицированное разделение

4. **Кодирование последовательностей**:
   - One-hot (4D)
   - k-mer частоты
   - Embeddings (DNABERT)

## 4. Особенности реализации

### Параметры загрузки
- `subset=True`: Ограничение размера датасета
- `sequence_length=2048`: Фиксация длины последовательностей
- `cache_dir='./'`: Локальное кэширование данных

### Структура DataFrame
| Колонка       | Описание                          | Пример        |
|---------------|-----------------------------------|---------------|
| sequence      | ДНК-последовательность           | "ATCG..."     |
| chromosome    | Хромосома (hg19)                  | "chr7"        |
| label_start   | Начальная позиция (0-based)       | 124535        |
| label_stop    | Конечная позиция                  | 124735        |
| label         | 0=промотор, 1=энхансер           | 1             |



In [1]:
# скачиваем данные формата fasta для human genome 38
# ! wget https://hgdownload.soe.ucsc.edu/goldenPath/hg38/bigZips/hg38.fa.gz

In [2]:
#!pip3 install pyfaidx -q
#!pip3 install Bio -q
#!pip3 install "datasets<=2.18.0" -q

import ssl
ssl._create_default_https_context = ssl._create_unverified_context

from datasets import load_dataset
import numpy as np
from Bio.Seq import Seq
import torch
from torch.utils.data import Dataset, DataLoader, random_split
import pandas as pd

# Загружаем датасет "genomics-long-range-benchmark" с Hugging Face.
# Параметры:
# - subset=True: использовать поднабор данных (как определено в скрипте датасета),
# - task_name="regulatory_element_promoter": выбрать конкретную задачу (промоторы),
# - sequence_length=2048: обрезать/паддить последовательности до длины 2048,
# - cache_dir_root='.': кэшировать данные и вспомогательные файлы в текущей директории.
promoter_ds = load_dataset("InstaDeepAI/genomics-long-range-benchmark", 
                           subset=True, 
                           task_name="regulatory_element_promoter", 
                           sequence_length=2048,
                           cache_dir_root='.')

# Преобразуем тренировочную часть датасета (первые 500000 объектов)
# в pandas DataFrame. Внутри это, как правило, словарь с полями
# вроде "sequence", "label" и т.п., который pandas превращает в таблицу.
df = pd.DataFrame(promoter_ds['train'][:500000])

# Сохраняем полученный DataFrame в текстовый файл 'reg_elements.csv'
# в табличном (tab-separated) формате:
# - sep='\t': использовать табуляцию как разделитель столбцов,
# - header=False: не записывать имена столбцов,
# - index=False: не записывать индекс DataFrame.
# Несмотря на расширение .csv, формат фактически TSV и по структуре
# может быть использован как исходник для дальнейшего формирования BED-подобного файла.
df.to_csv('reg_elements.csv', sep='\t', header=False, index=False)

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/datasets/load.py:1461: FutureWarning: The repository for InstaDeepAI/genomics-long-range-benchmark contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/InstaDeepAI/genomics-long-range-benchmark
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(


In [33]:
df['labels'].mean()

np.float64(0.048460484604846046)

## 2. Кодирование последовательностей <a name="кодирование-последовательностей"></a>

### Методы представления ДНК

#### 1. One-Hot Encoding

**Теория:**
Каждый нуклеотид представляется 4-мерным бинарным вектором:
- A = [1,0,0,0]
- T = [0,1,0,0]
- C = [0,0,1,0]
- G = [0,0,0,1]
- N (неизвестный) = [0,0,0,0]

**Реализация:**

```python
import numpy as np

def one_hot_encode(sequence, seq_length=1000):
    """Кодирование последовательности в one-hot формате"""
    base_dict = {'A':0, 'T':1, 'C':2, 'G':3}
    one_hot = np.zeros((seq_length, 4), dtype=np.float32)
    
    for i, base in enumerate(sequence[:seq_length]):
        if base in base_dict:
            one_hot[i, base_dict[base]] = 1
            
    return one_hot

# Пример использования
seq = "ATCGNNAT"
encoded = one_hot_encode(seq, seq_length=8)
print(encoded.shape)  # (8, 4)
```

#### 2. K-mer Frequency

**Теория:**
- Разбиваем последовательность на перекрывающиеся подстроки длины k
- Считаем частоту каждого k-mer

**Реализация:**

```python
from sklearn.feature_extraction.text import CountVectorizer

def get_kmer_frequencies(sequences, k=3):
    """Вычисление частот k-mer для списка последовательностей"""
    kmers = [get_kmers(seq, k) for seq in sequences]
    vectorizer = CountVectorizer(analyzer='char', ngram_range=(k,k))
    X = vectorizer.fit_transform(kmers)
    return X.toarray(), vectorizer.vocabulary_

def get_kmers(sequence, k=3):
    """Генерация k-mer для одной последовательности"""
    return ' '.join([sequence[i:i+k] for i in range(len(sequence)-k+1)])

# Пример
seqs = ["ATCGTA", "GCTAGC"]
kmer_freq, vocab = get_kmer_frequencies(seqs, k=2)
print("Vocabulary:", vocab)
print("Features:", kmer_freq)
```

#### 3. Embeddings

**Теория:**
Обученные векторные представления нуклеотидов или k-mer

**Реализация:**

```python
import torch
import torch.nn as nn

class DNAEmbedding(nn.Module):
    def __init__(self, vocab_size=4, embedding_dim=16):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        
    def forward(self, x):
        return self.embedding(x)

# Пример использования
embedder = DNAEmbedding()
input_seq = torch.LongTensor([0,1,2,3])  # A,T,C,G
embedded = embedder(input_seq)
print(embedded.shape)  # (4, 16)
```
---

**Задача 3. Кодирование**  
1. Какой размер будет у one-hot матрицы для последовательности длиной 100 п.н.?  
2. Почему k=3 часто используют для ДНК?  

---

**Задача 4. Создание датасета и DataLoader**  

In [3]:
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

import pandas as pd
import numpy as np

from tqdm.notebook import tqdm


class DNASequenceDataset(Dataset):
    def __init__(self, sequences, labels, seq_length=2048):
        """
        Args:
            sequences (list): List of DNA sequences
            labels (list): Corresponding labels
            seq_length (int): Fixed length for padding/truncating
        """

        # Метки классов переводим в тензор размера (N, 1), где N — число объектов
        self.labels = torch.tensor(labels).reshape(-1, 1)
        self.seq_length = seq_length

        # Словарь для one-hot кодирования нуклеотидов:
        # A, T, C, G → вектор длины 4, N → вектор из нулей (неопределённый нуклеотид)
        self.base_map = {'A': [1,0,0,0],
                        'T': [0,1,0,0],
                        'C': [0,0,1,0],
                        'G': [0,0,0,1],
                        'N': [0,0,0,0]}

        # Преобразуем все последовательности в one-hot тензоры и складываем в один
        # тензор размеров (N, seq_length, 4). tqdm показывает прогресс по последовательностям.
        self.features = torch.stack([
            self._encode_sequence(seq)
            for seq in tqdm(sequences)
        ])

    def __len__(self):
        # Количество объектов в датасете
        return len(self.labels)

    def _encode_sequence(self, seq):
        # Очистка последовательности от «левых» символов
        seq = self._clean_sequence(seq)
        # Приведение к фиксированной длине (обрезка или добивка N до seq_length)
        seq = self._pad_sequence(seq)

        # Перевод строки нуклеотидов в список символов,
        # маппинг по base_map и преобразование в тензор float32
        return torch.FloatTensor(pd.Series(list(seq)).map(self.base_map).tolist())

    def __getitem__(self, idx):
        # Возвращаем (features, label) по индексу для работы с DataLoader
        return self.features[idx], self.labels[idx]

    def _pad_sequence(self, seq):
        # Если последовательность длиннее нужной длины — обрезаем
        if len(seq) > self.seq_length:
            return seq[:self.seq_length]
        # Если короче — дополняем символами 'N' справа до seq_length
        return seq.ljust(self.seq_length, 'N')  # Pad with Ns

    def _clean_sequence(self, seq):
        # Заменяем все символы, не входящие в {A, T, C, G}, на 'N'
        valid_bases = {'A', 'T', 'C', 'G'}
        return ''.join([b if b in valid_bases else 'N' for b in seq])

    @staticmethod  # функция не привязана к конкретному объекту класса
    def collate_fn(batch):
        # batch — список кортежей (features, label)
        # Собираем батч признаков в один тензор размера (B, seq_length, 4)
        features = torch.stack([item[0] for item in batch])
        # Метки конкатенируем в тензор размера (B, 1)
        labels = torch.cat([item[1] for item in batch])
        return features, labels


# Разделение данных на train/val по столбцам df['sequence'] и df['labels']
X_train, X_val, y_train, y_val = train_test_split(df['sequence'], df['labels'], test_size=0.2)

# Создание датасетов из numpy‑массивов последовательностей и меток
train_dataset = DNASequenceDataset(X_train.values, y_train.values)
val_dataset = DNASequenceDataset(X_val.values, y_val.values)

# DataLoader’ы для обучения и валидации:
# batch_size=32, train — с перемешиванием, val — без перемешивания.
# Используем кастомный collate_fn для правильной сборки батчей.
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=DNASequenceDataset.collate_fn)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, collate_fn=DNASequenceDataset.collate_fn)

  0%|          | 0/79999 [00:00<?, ?it/s]

  0%|          | 0/20000 [00:00<?, ?it/s]

## 3. Архитектуры моделей <a name="построение-моделей"></a>

### Теоретическая основа

#### 1. Сверточные сети (CNN)
**Принцип работы:**
- Используют фильтры (ядра), скользящие по последовательности
- Выявляют локальные мотивы ДНК независимо от позиции
- Иерархическая структура: от простых к сложным паттернам

```python
import torch
import torch.nn as nn
import torch.nn.functional as F

class DNA_CNN(nn.Module):
    def __init__(self, seq_length=1000, num_classes=2):
        super().__init__()
        # 1D свертки для анализа последовательности
        self.conv_layers = nn.Sequential(
            nn.Conv1d(4, 32, kernel_size=9, padding=4),  # [batch, 32, seq_length]
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(32, 64, kernel_size=7, padding=3),
            nn.ReLU(),
            nn.MaxPool1d(2)
        )
        
        # Полносвязные слои
        self.fc_layers = nn.Sequential(
            nn.Linear(64 * (seq_length//4), 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = x.permute(0, 2, 1)  # [batch, channels, seq]
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)  # flatten
        return self.fc_layers(x)
```

#### 2. Рекуррентные сети (LSTM)
**Принцип работы:**
- Обрабатывают последовательность поэлементно
- Сохраняют "память" через скрытые состояния
- Ворота контролируют поток информации

```python
class DNA_LSTM(nn.Module):
    def __init__(self, input_dim=4, hidden_dim=64, num_layers=2, num_classes=2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            bidirectional=True,
            dropout=0.2 if num_layers > 1 else 0
        )
        self.attention = nn.Sequential(
            nn.Linear(hidden_dim*2, 1),
            nn.Softmax(dim=1)
        )
        self.classifier = nn.Linear(hidden_dim*2, num_classes)

    def forward(self, x):
        # x shape: [batch, seq_len, features]
        lstm_out, _ = self.lstm(x)  # [batch, seq_len, hidden_dim*2]
        
        # Механизм внимания
        attn_weights = self.attention(lstm_out)
        context = torch.sum(attn_weights * lstm_out, dim=1)
        
        return self.classifier(context)
```
---

### Сравнение подходов

| Критерий       | CNN               | LSTM              | Transformer       |
|----------------|-------------------|-------------------|-------------------|
| **Обработка последовательности** | Локальные окна | Пошаговая | Вся последовательность сразу |
| **Память**     | Нет               | Долгосрочная      | Глобальный контекст |
| **Параметры**  | Умеренно          | Много             | Очень много       |
| **Скорость**   | Быстро            | Медленно          | Зависит от реализации |
| **Лучшее применение** | Детекция мотивов | Анализ дальних зависимостей | Комплексные паттерны |

**Задача 5. Архитектуры**  
1. Почему CNN хорошо подходит для ДНК?  
2. Какой слой в LSTM помогает запоминать длинные последовательности?  
3. Как трансформеры учитывают порядок нуклеотидов без рекуррентности?
4. Почему в CNN для ДНК используют 1D свертки вместо 2D?

**Задача 6. Сверточная модель на все случаи жизни**  

In [12]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class FlexibleCNN(nn.Module):
    def __init__(
        self,
        input_channels=4,          # Для one-hot кодирования ДНК: A,T,C,G → 4 канала
        seq_length=2048,           # Длина входной последовательности (число позиций)
        conv_layers_config=[       # Список описаний сверточных слоёв (каждый — dict)
            {'out_channels': 32, 'kernel_size': 9, 'stride': 1, 'padding': 4},
            {'out_channels': 64, 'kernel_size': 7, 'stride': 1, 'padding': 3}
        ],
        pooling='max',             # Тип пулинга после каждой свёртки: 'max' или 'avg'
        pool_sizes=[2, 2],         # Размер окна пулинга для каждого сверточного блока
        activation='ReLU',         # Тип активации: ReLU, LeakyReLU, GELU, SELU, Swish
        normalization=None,        # Нормализация: 'batch', 'layer' или None (без нормализации)
        dropout=0.3,               # Вероятность дропаута в полносвязных слоях (0 — отключить)
        hidden_dims=[128],         # Размерности скрытых полносвязных слоёв
        num_classes=2              # Число классов на выходе (логиты размером [B, num_classes])
    ):
        super().__init__()

        # Список модулей, из которых последовательно собираются сверточные блоки
        self.conv_layers = nn.ModuleList()
        in_channels = input_channels
        current_seq_length = seq_length  # будем отслеживать длину последовательности после пулинга

        for i, config in enumerate(conv_layers_config):

            # 1D‑свёртка по последовательности: [B, in_channels, L] → [B, out_channels, L']
            self.conv_layers.append(
                nn.Conv1d(
                    in_channels=in_channels,
                    out_channels=config['out_channels'],
                    kernel_size=config['kernel_size'],
                    stride=config['stride'],
                    padding=config['padding']
                )
            )

            # Нелинейность после свёртки
            self.conv_layers.append(self._get_activation(activation))

            # При необходимости — нормализация по каналу/позициям
            if normalization == 'batch':
                # BatchNorm по каналам: [B, C, L]
                self.conv_layers.append(nn.BatchNorm1d(config['out_channels']))
            elif normalization == 'layer':
                # LayerNorm по всему тензору (C, L) для каждой позиции батча
                self.conv_layers.append(nn.LayerNorm([config['out_channels'], current_seq_length]))

            # Пулинг по длине последовательности для уменьшения размерности и выделения инвариантных признаков
            if pooling == 'max':
                self.conv_layers.append(nn.MaxPool1d(pool_sizes[i]))
            else:
                self.conv_layers.append(nn.AvgPool1d(pool_sizes[i]))

            # Обновляем длину последовательности после пулинга
            current_seq_length = current_seq_length // pool_sizes[i]
            # Количество каналов на выходе этого блока — вход для следующего
            in_channels = config['out_channels']

        # Полносвязные (FC) слои
        self.fc_layers = nn.ModuleList()
        # Вход в первый FC: все каналы всех позиций, "сплющенные" в вектор
        in_features = in_channels * current_seq_length
        # nn.Flatten() здесь не используется, вместо него — ручной view в forward

        for hidden_dim in hidden_dims:
            # Линейное преобразование: [B, in_features] → [B, hidden_dim]
            self.fc_layers.append(nn.Linear(in_features, hidden_dim))
            # Активация
            self.fc_layers.append(self._get_activation(activation))
            # Дропаут для регуляризации, если включен
            if dropout > 0:
                self.fc_layers.append(nn.Dropout(dropout))
            # Следующий FC будет получать на вход уже hidden_dim
            in_features = hidden_dim

        # Финальный классификатор: проекция в пространство классов
        self.classifier = nn.Linear(in_features, num_classes)

    def _get_activation(self, name):
        # Вспомогательная функция: по имени возвращает нужную активацию
        activations = {
            'ReLU': nn.ReLU(),
            'LeakyReLU': nn.LeakyReLU(0.1),
            'GELU': nn.GELU(),
            'SELU': nn.SELU(),
            'Swish': nn.SiLU()
        }
        # По умолчанию — ReLU, если имя не распознано
        return activations.get(name, nn.ReLU())

    def forward(self, x):
        # Ожидаемый вход: x shape [batch, seq_len, channels] 
        # Conv1d в PyTorch работает с [batch, channels, seq_len], поэтому делаем permute
        x = x.permute(0, 2, 1)  # [batch, channels, seq_len]

        # Прогоняем через последовательность свёрточных/нормализационных/пулинг‑слоёв
        for layer in self.conv_layers:
            x = layer(x)

        # Выпрямляем тензор в вектор признаков: [B, C_out, L_out] → [B, C_out * L_out]
        x = x.view(x.size(0), -1)

        # Прогон через все FC‑слои с активациями/дропаутом
        for layer in self.fc_layers:
            x = layer(x)

        # Линейный слой‑классификатор, на выходе — логиты классов
        return self.classifier(x)

**Задача 7. Рекуррентная модель на все случаи жизни**  

In [13]:
class FlexibleRNN(nn.Module):
    def __init__(
        self,
        input_size=4,             # Размер вектора на одну позицию (one-hot по нуклеотидам)
        hidden_size=64,           # Размер скрытого состояния RNN
        num_layers=2,             # Количество слоёв RNN (стек LSTM/GRU)
        rnn_type='LSTM',          # Тип рекуррентного блока: 'LSTM' или 'GRU'
        bidirectional=True,       # Использовать двунаправленную RNN (вперёд + назад)
        attention=True,           # Включить механизм внимания поверх выходов RNN
        normalization=None,       # Тип нормализации скрытых состояний: 'batch' или 'layer'
        dropout=0.2,              # Dropout между слоями RNN (если num_layers > 1)
        fc_dropout=0.3,           # Dropout перед финальным классификатором
        activation='ReLU',        # Нелинейность перед классификатором
        num_classes=2             # Число классов (размерность выходных логитов)
    ):
        super().__init__()

        # Выбор класса RNN в зависимости от rnn_type
        rnn_class = nn.LSTM if rnn_type == 'LSTM' else nn.GRU
        self.rnn = rnn_class(
            input_size=input_size,             # размер входного вектора на шаге
            hidden_size=hidden_size,           # размер скрытого состояния h_t
            num_layers=num_layers,             # число слоёв в стеке
            bidirectional=bidirectional,       # двунаправленная или нет
            dropout=dropout if num_layers > 1 else 0,  # dropout между слоями (не внутри слоя)
            batch_first=True                   # формат входа: [batch, seq_len, input_size]
        )

        # Нормализация по последнему измерению (каналы/фичи)
        self.norm = None
        if normalization == 'batch':
            # BatchNorm1d ожидает ввод [batch, features, seq_len], поэтому во forward будут permute
            self.norm = nn.BatchNorm1d(hidden_size * (2 if bidirectional else 1))
        elif normalization == 'layer':
            # LayerNorm по размерности признаков (hidden_size * num_directions)
            self.norm = nn.LayerNorm(hidden_size * (2 if bidirectional else 1))

        # Механизм внимания (self-attention по временной оси)
        self.attention = None
        if attention:
            # Линейный слой приводит каждый временной вектор к скаляру (логит веса),
            # затем Softmax по seq_len даёт распределение внимания для каждого элемента последовательности
            self.attention = nn.Sequential(
                nn.Linear(hidden_size * (2 if bidirectional else 1), 1),
                nn.Softmax(dim=1)
            )

        # Полносвязная часть (head классификатора)
        self.activation = self._get_activation(activation)
        # Dropout перед классификатором, если задан fc_dropout > 0
        self.dropout = nn.Dropout(fc_dropout) if fc_dropout > 0 else None
        # Линейный классификатор, принимающий контекстный вектор и выдающий логиты классов
        self.classifier = nn.Linear(
            hidden_size * (2 if bidirectional else 1),
            num_classes
        )

    def _get_activation(self, name):
        # Подбор функции активации по строковому имени
        activations = {
            'ReLU': nn.ReLU(),
            'LeakyReLU': nn.LeakyReLU(0.1),
            'GELU': nn.GELU(),
            'SELU': nn.SELU(),
            'Tanh': nn.Tanh()
        }
        # Если указан неизвестный тип, по умолчанию используется ReLU
        return activations.get(name, nn.ReLU())

    def forward(self, x):
        # Ожидаемый формат x: [batch, seq_len, input_size]
        # rnn_out: все скрытые состояния по шагам: [batch, seq_len, hidden_size * num_directions]
        # _ — кортеж (h_n, c_n) для LSTM или h_n для GRU, здесь не используется
        rnn_out, _ = self.rnn(x)

        # Нормализация по размерности признаков, если включена
        if self.norm is not None:
            # Для BatchNorm1d: переставляем оси в [batch, features, seq_len], применяем нормализацию,
            # затем возвращаемся к [batch, seq_len, features]
            rnn_out = self.norm(rnn_out.permute(0, 2, 1)).permute(0, 2, 1)

        # Механизм внимания
        if self.attention is not None:
            # Получаем веса внимания для каждого шага последовательности: [batch, seq_len, 1]
            attn_weights = self.attention(rnn_out)
            # Взвешенная сумма скрытых состояний по времени → контекстный вектор
            # [batch, seq_len, features] * [batch, seq_len, 1] → [batch, features]
            context = torch.sum(attn_weights * rnn_out, dim=1)
        else:
            # Без внимания берём скрытое состояние последнего шага последовательности
            # (последний time step по оси seq_len)
            context = rnn_out[:, -1, :]

        # Нелинейная активация над контекстом
        context = self.activation(context)

        # Дропаут как регуляризация перед классификацией
        if self.dropout is not None:
            context = self.dropout(context)

        # Линейный слой-классификатор, возвращает логиты классов [batch, num_classes]
        return self.classifier(context)

### Таблица гиперпараметров

| Параметр         | CNN Варианты                          | RNN Варианты                     |
|------------------|---------------------------------------|----------------------------------|
| **Активация**    | ReLU, LeakyReLU, GELU, SELU, Swish   | ReLU, LeakyReLU, GELU, Tanh     |
| **Нормализация** | BatchNorm1d, LayerNorm, None         | BatchNorm1d, LayerNorm, None    |
| **Дропаут**      | 0-0.5 (после каждого FC)             | 0-0.5 (между RNN слоями и FC)   |
| **Ядра**         | 3-11 (нечетные)                      | -                                |
| **Слои**         | 1-5 сверточных + 1-3 FC              | 1-5 RNN слоев                   |
| **Направление**  | -                                    | Uni/Bidirectional               |
| **Тип RNN**      | -                                    | LSTM, GRU                       |

In [14]:
# Определение устройства:
# если доступна GPU с поддержкой CUDA, используем её, иначе — CPU.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Пример конфигурации сверточной модели для ДНК:
# - два сверточных блока: 4→32 каналов (kernel=9), затем 32→64 (kernel=5),
# - LeakyReLU в качестве активации,
# - BatchNorm1d после свёрток,
# - dropout=0.4 в полносвязных слоях,
# - два FC-слоя: 128 → 64 перед финальным классификатором.
cnn_model = FlexibleCNN(
    conv_layers_config=[
        {'out_channels': 32, 'kernel_size': 9, 'stride': 1, 'padding': 4},
        {'out_channels': 64, 'kernel_size': 5, 'stride': 1, 'padding': 2}
    ],
    activation='LeakyReLU',
    normalization='batch',
    dropout=0.4,
    hidden_dims=[128, 64]
).to(device)  # переносим модель на выбранное устройство (CPU/GPU)

# Пример конфигурации рекуррентной модели:
# - трёхслойный двунаправленный LSTM (hidden_size=128),
# - включён механизм внимания по временной оси,
# - layer normalization по скрытому представлению,
# - dropout=0.3 между RNN-слоями,
# - fc_dropout=0.5 перед финальным классификатором,
# - GELU в качестве активации.
rnn_model = FlexibleRNN(
    rnn_type='LSTM',
    hidden_size=128,
    num_layers=3,
    bidirectional=True,
    attention=True,
    normalization='layer',
    dropout=0.3,
    fc_dropout=0.5,
    activation='GELU'
).to(device)  # также переносим на device

# Выводим текстовое представление архитектур обеих моделей
print("CNN Model:")
print(cnn_model)

print("\nRNN Model:")
print(rnn_model)


CNN Model:
FlexibleCNN(
  (conv_layers): ModuleList(
    (0): Conv1d(4, 32, kernel_size=(9,), stride=(1,), padding=(4,))
    (1): LeakyReLU(negative_slope=0.1)
    (2): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (3): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv1d(32, 64, kernel_size=(5,), stride=(1,), padding=(2,))
    (5): LeakyReLU(negative_slope=0.1)
    (6): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (7): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (fc_layers): ModuleList(
    (0): Linear(in_features=32768, out_features=128, bias=True)
    (1): LeakyReLU(negative_slope=0.1)
    (2): Dropout(p=0.4, inplace=False)
    (3): Linear(in_features=128, out_features=64, bias=True)
    (4): LeakyReLU(negative_slope=0.1)
    (5): Dropout(p=0.4, inplace=False)
  )
  (classifier): Linear(in_features=64, out_features=2, bias=True)
)

R

## 4. Обучение и оптимизация <a name="обучение-и-оптимизация"></a>  

### Теоретическая основа

**Процесс обучения CNN для ДНК:**
1. **Особенности задачи:**
   - Высокая размерность входных данных (длинные последовательности)
   - Часто несбалансированные классы (разное количество промоторов/энхансеров)
   - Важность интерпретируемости (какие участки ДНК важны для классификации)

2. **Ключевые компоненты:**
   - Оптимизатор: адаптивно изменяет веса модели
   - Scheduler: динамически регулирует learning rate
   - Регуляризация: предотвращает переобучение (dropout, batch norm)

**Задача 8. Инициализация обучения FlexCNN**  

In [15]:
import torch
import torch.optim as optim
from torch.optim import lr_scheduler

# Словарь с различными оптимизаторами для одной и той же модели (cnn_model):
# - Adam: стандартный адаптивный оптимизатор, умеренный weight_decay для регуляризации;
# - AdamW: модификация Adam с корректным decoupled weight decay (часто лучше для нейросетей);
# - SGD: стохастический градиентный спуск с моментом (классический вариант).
optimizers = {
    'Adam': optim.Adam(cnn_model.parameters(), lr=1e-3, weight_decay=1e-4),
    'AdamW': optim.AdamW(cnn_model.parameters(), lr=5e-4, weight_decay=1e-3),
    'SGD': optim.SGD(cnn_model.parameters(), lr=0.01, momentum=0.9)
}

# Словарь с различными scheduler'ами для изменения learning rate:
# Здесь все три scheduler'а привязаны к оптимизатору 'Adam'.
schedulers = {
    # StepLR: каждые step_size эпох уменьшает lr в gamma раз (ступенчатое снижение).
    'StepLR': lr_scheduler.StepLR(optimizers['Adam'], step_size=5, gamma=0.1),

    # ReduceLROnPlateau: уменьшает lr, если целевая метрика (mode='max') не улучшается
    # в течение 'patience' шагов (например, по валидационной точности).
    'ReduceLROnPlateau': lr_scheduler.ReduceLROnPlateau(
        optimizers['Adam'],
        mode='max',
        patience=3,
        factor=0.5,
    ),

    # CosineAnnealingLR: косинусное «затухание» lr от начального значения до eta_min
    # за T_max эпох.
    'Cosine': lr_scheduler.CosineAnnealingLR(
        optimizers['Adam'],
        T_max=10,
        eta_min=1e-6
    )
}

# Функция потерь для многоклассовой классификации — CrossEntropyLoss.
# Аргумент weight задаёт веса классов: здесь класс 1 штрафуется сильнее (в 2 раза),
# что полезно при дисбалансе классов. Веса предварительно переносим на device (CPU/GPU).
criterion = nn.CrossEntropyLoss(
    weight=torch.tensor([1.0, 2.0]).to(device)  # Для несбалансированных классов
)

Using device: cpu


**Задача 9. Функции обучения и валидации**

In [16]:
def train_epoch(model, dataloader, optimizer, criterion, device):
    model.train()          # Переводим модель в режим обучения (включает dropout, BatchNorm в train-режиме)
    running_loss = 0.0     # Суммарный loss за эпоху
    correct = 0            # Количество верных предсказаний
    total = 0              # Общее количество примеров

    for inputs, labels in dataloader:
        # Перенос батча данных и меток на выбранное устройство (CPU/GPU)
        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()      # Обнуляем накопленные градиенты

        outputs = model(inputs)    # Прямой проход: [batch, num_classes]
        loss = criterion(outputs, labels)  # Вычисляем loss для данного батча
        loss.backward()            # Обратное распространение ошибки (расчёт градиентов)
        optimizer.step()           # Обновление параметров модели

        running_loss += loss.item()    # Накопление значения loss
        _, predicted = torch.max(outputs.data, 1)  # Индексы классов с максимальным логитом
        total += labels.size(0)                     # Увеличиваем счётчик всех примеров
        correct += (predicted == labels).sum().item()  # Увеличиваем счётчик верных предсказаний

    epoch_loss = running_loss / len(dataloader)  # Средний loss по батчам
    epoch_acc = correct / total                  # Доля верных предсказаний за эпоху

    return epoch_loss, epoch_acc


def validate(model, dataloader, criterion, device):
    model.eval()           # Переводим модель в режим оценки (отключает dropout, BatchNorm в eval-режиме)
    running_loss = 0.0
    correct = 0
    total = 0

    # Валидация без вычисления градиентов (экономия памяти и времени)
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs)             # Прямой проход
            loss = criterion(outputs, labels)   # Loss на валидационном батче

            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    val_loss = running_loss / len(dataloader)  # Средний валидационный loss
    val_acc = correct / total                  # Валидационная точность

    return val_loss, val_acc

**Задача 10. Полный цикл обучения с логированием**

In [23]:
# !pip3 install mlflow -q
import mlflow

def train_model(
    model,
    train_loader,
    val_loader,
    optimizer,
    scheduler,
    criterion,
    device,
    epochs=20,
    use_mlflow=True
):

    best_val_acc = 0.0
    # Словарь для хранения истории обучения по эпохам
    history = {
        'train_loss': [],
        'train_acc': [],
        'val_loss': [],
        'val_acc': [],
        'lr': []
    }

    for epoch in tqdm(range(epochs)):
        # Один проход обучения по train_loader: считаем средний loss и accuracy
        train_loss, train_acc = train_epoch(
            model, train_loader, optimizer, criterion, device
        )
        # Один проход валидации по val_loader без градиентов
        val_loss, val_acc = validate(
            model, val_loader, criterion, device
        )

        # Обновление scheduler'а:
        # для ReduceLROnPlateau передаём метрику (здесь val_acc),
        # для остальных вызываем step() просто по номеру эпохи.
        if isinstance(scheduler, lr_scheduler.ReduceLROnPlateau):
            scheduler.step(val_acc)
        else:
            scheduler.step()

        # Текущий learning rate берём из первой группы параметров оптимизатора
        current_lr = optimizer.param_groups[0]['lr']

        # Сохраняем значения метрик в историю
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['lr'].append(current_lr)

        # Лог в консоль по итогам эпохи
        print(f"Epoch {epoch+1}/{epochs}:")
        print(f"  Train Loss: {train_loss:.4f} | Acc: {train_acc:.4f}")
        print(f"  Val Loss: {val_loss:.4f} | Acc: {val_acc:.4f}")
        print(f"  LR: {current_lr:.6f}")

        if use_mlflow:
            # Логирование метрик в MLflow для текущей эпохи.
            # step=epoch+1 позволяет потом строить графики по шагам.
            mlflow.log_metrics(
                {
                    "train_loss": train_loss,
                    "train_acc": train_acc,
                    "val_loss": val_loss,
                    "val_acc": val_acc,
                    "lr": current_lr,
                    "epoch": epoch + 1,
                },
                step=epoch + 1,
            )

        # Сохранение лучшей модели по val_acc:
        # если текущая точность выше предыдущего максимума — обновляем best_val_acc
        # и сохраняем state_dict в файл 'best_model.pth'.
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), 'best_model.pth')
            if use_mlflow:
                # Логируем сохранённый чекпоинт как артефакт в MLflow
                mlflow.log_artifact('best_model.pth', artifact_path="checkpoints")

    # Возвращаем историю метрик по эпохам для дальнейшего анализа/визуализации
    return history


**Задача 11 Визуализация результатов**

In [24]:
import matplotlib.pyplot as plt

def plot_training_history(history):
    plt.figure(figsize=(12, 4))

    # Loss
    plt.subplot(1, 2, 1)
    plt.plot(history['train_loss'], label='Train')
    plt.plot(history['val_loss'], label='Validation')
    plt.title('Loss')
    plt.xlabel('Epoch')
    plt.legend()

    # Accuracy
    plt.subplot(1, 2, 2)
    plt.plot(history['train_acc'], label='Train')
    plt.plot(history['val_acc'], label='Validation')
    plt.title('Accuracy')
    plt.xlabel('Epoch')
    plt.legend()

    plt.tight_layout()
    plt.show()

    # Learning rate
    plt.figure(figsize=(6, 4))
    plt.plot(history['lr'])
    plt.title('Learning Rate Schedule')
    plt.xlabel('Epoch')
    plt.ylabel('LR')
    plt.show()

In [25]:
# Конфигурация эксперимента
config = {
    "architecture": "FlexibleCNN",
    "conv_layers": "32x9, 64x7",
    "activation": "LeakyReLU",
    "dropout": 0.3,
    "optimizer": "AdamW",
    "scheduler": "ReduceLROnPlateau",
    "batch_size": 32,
    "epochs": 20
}

# Запуск эксперимента в MLflow
mlflow.set_experiment("dna-classification")

with mlflow.start_run(run_name=f"CNN_{config['conv_layers']}"):
    mlflow.log_params(config)

    history = train_model(
        model=cnn_model,
        train_loader=train_loader,
        val_loader=val_loader,
        optimizer=optimizers['AdamW'],
        scheduler=schedulers['ReduceLROnPlateau'],
        criterion=criterion,
        device=device,
        epochs=config['epochs'],
        use_mlflow=True
    )

    # сохраняем модель локально и логируем как артефакт
    torch.save(cnn_model.state_dict(), "final_model.pth")
    mlflow.log_artifact("final_model.pth", artifact_path="models")

    plot_training_history(history)


Epoch 1/20:
  Train Loss: 0.1768 | Acc: 0.9638
  Val Loss: 0.1368 | Acc: 0.9719
  LR: 0.000500
Epoch 2/20:
  Train Loss: 0.1327 | Acc: 0.9717
  Val Loss: 0.1394 | Acc: 0.9688
  LR: 0.000500
Epoch 3/20:
  Train Loss: 0.0966 | Acc: 0.9774
  Val Loss: 0.1511 | Acc: 0.9717
  LR: 0.000500



KeyboardInterrupt



In [31]:
!mlflow ui --backend-store-uri sqlite:///mlflow.db --port 5000

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/pydantic/_internal/_fields.py:132: UserWarning: Field "model_name" in PromptModelConfig has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
Registry store URI not provided. Using backend store URI.
2026/02/17 20:14:37 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.schemas
2026/02/17 20:14:37 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.tables
2026/02/17 20:14:37 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.types
2026/02/17 20:14:37 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.constraints
2026/02/17 20:14:37 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.defaults
2026/02/17 20:14:37 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.comments
2026/02/17 20:14:37 INFO alembic.runtime.migration: Conte


### Интерпретация результатов

1. **Анализ графиков:**
   - Сходимость обучения (loss должен уменьшаться)
   - Разрыв между train/val (индикатор переобучения)
   - Динамика learning rate

2. **Метрики качества:**
   - Accuracy (общая точность)
   - Precision/Recall (для несбалансированных данных)
   - ROC-AUC (качество разделения классов)

3. **Оптимизация:**
   - Если train loss не уменьшается → увеличить LR или сложность модели
   - Если val loss растет → добавить регуляризацию
   - Если accuracy "застряла" → попробовать другой оптимизатор